1.p(a∣b)=0.4;
2.p(c∣b)=0.4

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 转小写，去标点
    text = re.sub(r'[^a-z\s]', '', text.lower())
    words = text.split()
    
    # 构建词汇表（按频率排序）
    vocab = {word: i for i, (word, _) in enumerate(sorted(Counter(words).items(), key=lambda x: -x[1]))}
    
    # 生成特征和标签
    features, labels = [], []
    for i in range(len(words) - n):
        features.append([vocab[words[i+j]] for j in range(n)])
        labels.append(vocab[words[i+n]])
    
    return vocab, features, labels

# 测试
text = "The time machine"
n = 2
vocab, features, labels = preprocess_text(text, n)
print("词汇表:", vocab)
print("特征:", features)
print("标签:", labels)

词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征: [[0, 1]]
标签: [2]


模型表达式：
h_t = W_hh * h_{t-1} + W_hx * x_t
o_t = W_oh * h_t
损失函数：
L = 1/2 * sum_{t=1}^T (o_t - y_t)^2
单时间步损失对隐藏状态梯度
dL_t /dh_t = (o_t - y_t)・W_oh^T ，简记为 δ_t
BPTT 递推梯度关系
dL /dh_τ = δ_τ + W_hh^T・dL /dh_{τ+1}
损失对权重 W_hh 的总梯度
dL /dW_hh = sum_{τ=0}^{T-1} [dL /dh_{τ+1}]・h_τ^T
梯度消失 / 爆炸判定：
设 ρ(W_hh) 为矩阵 W_hh 的谱半径（矩阵最大特征值的模）
梯度消失：ρ(W_hh) < 1，远距离梯度多次矩阵连乘后趋近于 0
梯度爆炸：ρ(W_hh) > 1，梯度随时间步指数级放大
临界状态：ρ(W_hh) = 1，梯度无明显消失或爆炸

In [6]:
import torch

def rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h):
    return torch.tanh(x_t @ W_xh.T + h_prev @ W_hh.T + b_h)

def rnn_cell_backward(dh_next, x_t, h_prev, W_hh, W_xh, b_h):
    h_t = torch.tanh(x_t @ W_xh.T + h_prev @ W_hh.T + b_h)
    dtanh = dh_next * (1 - h_t**2)
    
    dx_t = dtanh @ W_xh
    dh_prev = dtanh @ W_hh
    dW_xh = dtanh.T @ x_t
    dW_hh = dtanh.T @ h_prev
    db_h = dtanh.sum(dim=0)
    
    return dx_t, dh_prev, dW_hh, dW_xh, db_h

# 例子（batch_size=1）
x_t = torch.tensor([[1.0, 2.0]])
h_prev = torch.tensor([[0.0, 0.0]])
W_hh = torch.tensor([[0.5, 0.0], [0.0, 0.5]])
W_xh = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
b_h = torch.tensor([0.0, 0.0])

h_t = rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h)
print("隐藏状态:", h_t)  # tensor([[0.7616, 0.9640]])

dh_next = torch.tensor([[1.0, 1.0]])
dx_t, dh_prev, dW_hh, dW_xh, db_h = rnn_cell_backward(dh_next, x_t, h_prev, W_hh, W_xh, b_h)
print("dx_t:", dx_t)

隐藏状态: tensor([[0.7616, 0.9640]])
dx_t: tensor([[0.4200, 0.0707]])


已知条件：层数 L，单层单向隐藏单元数 H，输入维度 D，输出维度 O
单层单向 RNN 参数：HD + HH + 2H
单层双向 RNN 参数：2(HD + H² + 2H)
L 层双向循环层总参数：2L*(HD + H² + 2H)
输出全连接层参数：2H*O + O
总参数表达式：
Total = 2L(HD + H² + 2H) + 2HO + O

In [8]:
import torch
import torch.nn as nn

class BidirectionalRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, bidirectional=True, batch_first=False)
    
    def forward(self, X):
        output, _ = self.rnn(X)
        final_state = output[-1]
        return output, final_state

# 例子
seq_len, batch, input_dim, hidden_dim = 3, 1, 2, 2
X = torch.randn(seq_len, batch, input_dim)
model = BidirectionalRNN(input_dim, hidden_dim)
output, final = model(X)

print("输入形状:", X.shape)        # (3, 1, 2)
print("输出形状:", output.shape)   # (3, 1, 4)
print("最终状态:", final.shape)    # (1, 4)

输入形状: torch.Size([3, 1, 2])
输出形状: torch.Size([3, 1, 4])
最终状态: torch.Size([1, 4])


v_c：中心词向量，u_o：真实上下文词向量，u_nk：第 k 个负样本向量
sigmoid 函数：σ(z) = 1/(1+exp (-z))
K：负样本数量
单窗口负对数似然损失：
L = - [logσ(v_c^T・u_o) + sum_{k=1}^K log (1 - σ(v_c^T・u_nk)) ]
全局全部文本总损失：
L_total = - sum_{全部窗口 (w_c,w_o)} [ logσ(v_c^T・u_o) + sum_{k=1}^K log (1 - σ(v_c^T・u_nk)) ]
负样本采样方法：
依据噪声分布 P_n (w) 采样 K 个词汇；
工程通用噪声分布：P_n (w) 正比于 count (w)^(3/4)；
采样时排除当前真实上下文 w_o，保证负样本与正样本不重复。

In [9]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, target_indices, W, W_out):
    context_embeds = W[context_indices]
    h = context_embeds.mean(dim=1)
    scores = h @ W_out
    log_probs = F.log_softmax(scores, dim=1)
    loss = F.nll_loss(log_probs, target_indices)
    return loss

# 简单例子
V, d, context_size, batch = 5, 3, 2, 1
context_indices = torch.tensor([[0, 1]])
target_indices = torch.tensor([2])
W = torch.randn(V, d)
W_out = torch.randn(d, V)

loss = cbow_forward(context_indices, target_indices, W, W_out)
print("损失:", loss.item())

损失: 0.958501935005188


已知矩阵维度：
Q ∈ R^(2×4)，K ∈ R^(3×4)，V ∈ R^(3×5)，d_k=4，sqrt (d_k)=2
计算公式：
Score = Q・K^T /sqrt (d_k)
Output = softmax (Score)・V
分步计算：
将 K 转置得到 K^T ∈ R^(4×3)，计算原始相似度矩阵 S_raw = Q・K^T，维度 2×3；
缩放得到得分矩阵 S = S_raw / 2，维度 2×3；
对 S 逐行做 softmax 归一化得到权重矩阵 α：
α_ij = exp (S_ij) /sum_{m=1}^3 exp (S_im) ，α 维度 2×3；
使用 α 加权求和 V，得到最终输出矩阵 Output = α・V，维度 2×5。

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch, _ = X.shape
        
        # 线性投影并分头
        Q = self.W_q(X).reshape(seq_len, batch, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        K = self.W_k(X).reshape(seq_len, batch, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        V = self.W_v(X).reshape(seq_len, batch, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        # Q,K,V: (batch, num_heads, seq_len, d_k)
        
        # 缩放点积注意力
        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)  # (batch, num_heads, seq_len, seq_len)
        attn = F.softmax(scores, dim=-1)
        out = attn @ V  # (batch, num_heads, seq_len, d_k)
        
        # 拼接并投影
        out = out.transpose(1, 2).reshape(seq_len, batch, self.d_model)
        return self.W_o(out)

# 测试
seq_len, batch, d_model, num_heads = 5, 3, 4, 2
X = torch.randn(seq_len, batch, d_model)
model = MultiHeadAttention(d_model, num_heads)
output = model(X)
print("输入形状:", X.shape)
print("输出形状:", output.shape)  # (5, 3, 4)

输入形状: torch.Size([5, 3, 4])
输出形状: torch.Size([5, 3, 4])
